<a href="https://colab.research.google.com/github/yashaskiran/computer_vision_projects/blob/main/Event_based_vision_pipeline_for_high_speed_counter_UAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tonic -q

In [ ]:
!pip install h5py numpy matplotlib scipy requests

In [ ]:
!pip install h5py numpy matplotlib scipy -q

In [ ]:
import os
!pip install --quiet --force-reinstall numpy==1.26.4
os.kill(os.getpid(), 9)  # forces a hard runtime restart

In [ ]:
import numpy as np
print(np.__version__)  # should print 1.26.4

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import h5py
import requests
import os
from dataclasses import dataclass
from collections import deque
from scipy import ndimage

# =========================
# 1. SENSOR CONFIG
# =========================
@dataclass
class SensorConfig:
    width: int = 240
    height: int = 180
    time_window_us: int = 10_000
    refractory_period_us: int = 500
    noise_rate_hz: float = 0.1

# =========================
# 2. DOWNLOAD SAMPLE DATASET
# =========================
# =========================
# 2. LOAD DATASET VIA TONIC
# =========================
import tonic
import numpy as np

def load_tonic_dataset(save_dir: str = "./data"):
    dataset = tonic.datasets.DAVISDATA(
        save_to=save_dir,
        recording="shapes_6dof"   # valid recording name for DAVISDATA
    )
    data, targets = dataset[0]
    events, imu, images = data   # DAVISDATA returns (events, imu, images)

    xs = events["x"].astype(np.int64)
    ys = events["y"].astype(np.int64)
    ts = events["t"].astype(np.int64)
    ps = np.where(events["p"] == 1, 1, -1).astype(np.int64)

    print(f"Loaded: shapes_6dof")
    print(f"  Events   : {len(ts):,}")
    print(f"  Duration : {(ts.max() - ts.min()) / 1e6:.2f} s")
    print(f"  X range  : {xs.min()} – {xs.max()}")
    print(f"  Y range  : {ys.min()} – {ys.max()}")

    return xs, ys, ts, ps
# =========================
# 3. H5 EVENT STREAM
# =========================
class ArrayEventStream:
    """Wraps a pre-loaded numpy array into the same step() interface."""
    def __init__(self, xs, ys, ts, ps, cfg: SensorConfig):
        self.cfg     = cfg
        self.xs      = xs
        self.ys      = ys
        self.ts      = ts
        self.ps      = ps
        self.t_start = int(ts.min())
        self.t_end   = int(ts.max())

    def step(self, t0_us: int):
        t1_us = t0_us + self.cfg.time_window_us
        i0 = np.searchsorted(self.ts, t0_us, side="left")
        i1 = np.searchsorted(self.ts, t1_us, side="left")
        if i0 >= i1:
            return np.empty((0, 4), dtype=np.int64), []
        xs = np.clip(self.xs[i0:i1], 0, self.cfg.width  - 1)
        ys = np.clip(self.ys[i0:i1], 0, self.cfg.height - 1)
        ts = self.ts[i0:i1]
        ps = self.ps[i0:i1]
        return np.stack([xs, ys, ts, ps], axis=1).astype(np.int64), []

# =========================
# 4. TIME SURFACE
# =========================
class TimeSurface:
    def __init__(self, cfg: SensorConfig, decay_us: int = 30_000):
        self.cfg      = cfg
        self.decay_us = decay_us
        self.surface  = np.zeros((cfg.height, cfg.width), dtype=np.float64)
        self.last_t   = np.zeros((cfg.height, cfg.width), dtype=np.int64)

    def update(self, events: np.ndarray, current_t: int) -> np.ndarray:
        if len(events) == 0:
            return self.surface
        for ev in events:
            self.last_t[int(ev[1]), int(ev[0])] = int(ev[2])
        dt           = np.maximum(current_t - self.last_t, 0).astype(np.float64)
        self.surface = np.exp(-dt / self.decay_us)
        return self.surface

# =========================
# 5. EVENT CLUSTERER
# =========================
class EventClusterer:
    def __init__(self, threshold: float = 0.3, min_cluster_px: int = 20):
        self.threshold      = threshold
        self.min_cluster_px = min_cluster_px

    def detect(self, surface: np.ndarray) -> list:
        binary            = (surface > self.threshold).astype(np.uint8)
        labeled, n_labels = ndimage.label(binary)
        detections        = []
        for label_id in range(1, n_labels + 1):
            ys, xs = np.where(labeled == label_id)
            if len(xs) < self.min_cluster_px:
                continue
            x0, x1 = xs.min(), xs.max()
            y0, y1 = ys.min(), ys.max()
            detections.append({
                "cx": (x0 + x1) / 2.0, "cy": (y0 + y1) / 2.0,
                "w":  x1 - x0 + 1,     "h":  y1 - y0 + 1,
                "x0": x0, "y0": y0,     "x1": x1, "y1": y1,
                "n_pixels": len(xs)
            })
        return detections

# =========================
# 6. PIPELINE RUNNER
# =========================
class EventCameraPipeline:
    def __init__(self, cfg: SensorConfig, stream):
        self.cfg       = cfg
        self.stream    = stream
        self.surface   = TimeSurface(cfg)
        self.clusterer = EventClusterer()
        self.history   = deque(maxlen=100)

    def run(self, n_windows: int = 50):
        t_us = self.stream.t_start
        for i in range(n_windows):
            if t_us >= self.stream.t_end:
                print(f"End of dataset reached at window {i}.")
                break
            events, _ = self.stream.step(t_us)
            t_us     += self.cfg.time_window_us
            if len(events) == 0:
                continue
            surf = self.surface.update(events, t_us)
            dets = self.clusterer.detect(surf)
            self.history.append({
                "t_us":    t_us,
                "events":  events,
                "surface": surf.copy(),
                "dets":    dets,
            })
        return list(self.history)

# =========================
# 7. VISUALISATION
# =========================
def visualise_frame(frame, cfg, ax_events, ax_surface):
    events  = frame["events"]
    surface = frame["surface"]
    dets    = frame["dets"]
    t_ms    = frame["t_us"] / 1000.0

    ax_events.cla()
    ax_events.set_facecolor("black")
    if len(events):
        pos = events[events[:, 3] == +1]
        neg = events[events[:, 3] == -1]
        if len(pos): ax_events.scatter(pos[:, 0], pos[:, 1], s=0.4, c="cyan", alpha=0.6, linewidths=0)
        if len(neg): ax_events.scatter(neg[:, 0], neg[:, 1], s=0.4, c="red",  alpha=0.6, linewidths=0)
    ax_events.set_xlim(0, cfg.width)
    ax_events.set_ylim(cfg.height, 0)
    ax_events.set_title(f"Events  t={t_ms:.1f} ms", color="white", fontsize=8)
    ax_events.tick_params(colors="white")
    for spine in ax_events.spines.values(): spine.set_edgecolor("gray")

    for d in dets:
        rect = mpatches.Rectangle((d["x0"], d["y0"]), d["w"], d["h"],
                                   linewidth=1.5, edgecolor="lime", facecolor="none")
        ax_events.add_patch(rect)
        ax_events.text(d["x0"], d["y0"] - 2, f"({d['cx']:.0f},{d['cy']:.0f})",
                       color="lime", fontsize=6)

    ax_surface.cla()
    ax_surface.imshow(surface, cmap="inferno", vmin=0, vmax=1,
                      origin="upper", interpolation="nearest")
    ax_surface.set_title(f"Time Surface  t={t_ms:.1f} ms", fontsize=8)
    for d in dets:
        rect = mpatches.Rectangle((d["x0"], d["y0"]), d["w"], d["h"],
                                   linewidth=1.5, edgecolor="lime", facecolor="none")
        ax_surface.add_patch(rect)


def plot_pipeline_summary(history, cfg):
    n       = len(history)
    indices = [0, n // 2, n - 1]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.patch.set_facecolor("#111111")
    for col, idx in enumerate(indices):
        visualise_frame(history[idx], cfg, axes[0, col], axes[1, col])
    axes[0, 0].set_ylabel("ON/OFF Events", color="white", fontsize=9)
    axes[1, 0].set_ylabel("Time Surface",  fontsize=9)
    fig.suptitle("Event Camera Pipeline — Live H5 Dataset",
                 color="white", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig("pipeline_summary.png", dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.show()


def plot_detection_stats(history):
    times_ms = [f["t_us"] / 1000 for f in history]
    n_events = [len(f["events"]) for f in history]
    n_dets   = [len(f["dets"])   for f in history]
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
    fig.patch.set_facecolor("#111111")
    for ax in (ax1, ax2):
        ax.set_facecolor("#1a1a1a")
        ax.tick_params(colors="white")
        for spine in ax.spines.values(): spine.set_edgecolor("#444")
    ax1.fill_between(times_ms, n_events, alpha=0.7, color="cyan")
    ax1.plot(times_ms, n_events, color="cyan", lw=1.2)
    ax1.set_ylabel("Events / window", color="white")
    ax2.plot(times_ms, n_dets, color="lime", lw=1.8, marker="o", markersize=3)
    ax2.set_ylabel("Detections", color="white")
    ax2.set_xlabel("Time (ms)", color="white")
    if max(n_dets) > 0:
        ax2.set_ylim(-0.5, max(n_dets) + 1.5)
    ax2.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    fig.suptitle("Pipeline Statistics — Live Dataset", color="white",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig("detection_stats.png", dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.show()

# =========================
# 8. MAIN
# =========================
cfg = SensorConfig(width=240, height=180, time_window_us=10_000)

# Download + load dataset
xs, ys, ts, ps = load_tonic_dataset("./data")
stream = ArrayEventStream(xs, ys, ts, ps, cfg)

# Run pipeline
pipeline = EventCameraPipeline(cfg, stream)
print("\nRunning pipeline …")
history = pipeline.run(n_windows=50)

# Stats
print(f"\nFrames processed : {len(history)}")
total_ev = sum(len(f["events"]) for f in history)
print(f"Total events     : {total_ev:,}")
avg_dets = np.mean([len(f["dets"]) for f in history])
print(f"Avg detections   : {avg_dets:.2f} / window")

last = history[-1]
print(f"\nLast frame  t={last['t_us']/1000:.1f} ms")
for i, d in enumerate(last["dets"]):
    print(f"  [{i}]  centre=({d['cx']:.1f},{d['cy']:.1f})  "
          f"bbox={d['w']}×{d['h']} px")

plot_pipeline_summary(history, cfg)
plot_detection_stats(history)
print("\nDone.")